# Notebook 08 — Cross-Dataset Validation (CORRECTED)

## GSE136666 bulk RNA-seq × GSE184950 snRNA-seq pseudobulk

Esta versión corrige el problema de integración de identificadores que podía dejar sólo un gen (por ejemplo ACO2) o producir `cross_dataset_score = NaN`. La firma primaria y la tabla priorizada de Notebook 05 se usan como tabla maestra de identidad génica. No se concatenan matrices bulk y snRNA-seq.

In [ ]:
from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RANDOM_STATE=42
ALPHA=0.05
cwd=Path.cwd()
if (cwd/'data').exists(): PROJECT_ROOT=cwd
elif (cwd.parent/'data').exists(): PROJECT_ROOT=cwd.parent
else: PROJECT_ROOT=Path(r'C:\Users\Camfe\pythonlab')
DATA_DIR=PROJECT_ROOT/'data'; RESULTS_DIR=PROJECT_ROOT/'results'; FIGURES_DIR=PROJECT_ROOT/'figures'
NB08_RESULTS=RESULTS_DIR/'validation'/'notebook08'; NB08_FIGURES=FIGURES_DIR/'validation'/'notebook08'
NB08_RESULTS.mkdir(parents=True, exist_ok=True); NB08_FIGURES.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT:',PROJECT_ROOT)

## 1. Rutas de entrada

In [ ]:
def fallback_find(path, pattern):
    if path.exists(): return path
    hits=list(PROJECT_ROOT.rglob(pattern))
    return hits[0] if hits else path

PRIMARY_SIGNATURE_FILE=fallback_find(PROJECT_ROOT/'data'/'processed'/'notebook05'/'GSE136666_SN_primary_metabolic_signature.csv','GSE136666_SN_primary_metabolic_signature.csv')
PRIORITIZED_FILE=fallback_find(PROJECT_ROOT/'data'/'processed'/'notebook05'/'GSE136666_SN_metabolic_candidates_prioritized.csv','GSE136666_SN_metabolic_candidates_prioritized.csv')
BULK_DE_FILE=fallback_find(PROJECT_ROOT/'data'/'processed'/'notebook03'/'GSE136666_SN_DE_full_results.csv','GSE136666_SN_DE_full_results.csv')
NB07_INPUT=fallback_find(PROJECT_ROOT/'results'/'single_cell'/'notebook07'/'Notebook08_cross_dataset_input.csv','Notebook08_cross_dataset_input.csv')
for n,p in [('PRIMARY_SIGNATURE_FILE',PRIMARY_SIGNATURE_FILE),('PRIORITIZED_FILE',PRIORITIZED_FILE),('BULK_DE_FILE',BULK_DE_FILE),('NB07_INPUT',NB07_INPUT)]: print(n,p,'exists=',p.exists())
missing=[n for n,p in [('Notebook05 primary signature',PRIMARY_SIGNATURE_FILE),('Notebook05 prioritized candidates',PRIORITIZED_FILE),('Notebook03 bulk DE',BULK_DE_FILE),('Notebook07 input',NB07_INPUT)] if not p.exists()]
if missing: raise FileNotFoundError('Missing required files: '+', '.join(missing))

## 2. Funciones auxiliares de columnas e identificadores

In [ ]:
def detect_column(df, preferred, contains=None, required=False, label='column'):
    lm={c.lower():c for c in df.columns}
    for x in preferred:
        if x.lower() in lm: return lm[x.lower()]
    if contains:
        for c in df.columns:
            cl=c.lower()
            if all(k.lower() in cl for k in contains): return c
    if required: raise KeyError(f'Could not detect {label}. Columns: {df.columns.tolist()}')
    return None

def norm_symbol(s):
    return s.astype(str).str.strip().str.upper().replace({'NAN':np.nan,'NONE':np.nan,'':np.nan})

def num(s): return pd.to_numeric(s, errors='coerce')

## 3. Cargar firma primaria de Notebook 05

In [ ]:
primary_sig=pd.read_csv(PRIMARY_SIGNATURE_FILE)
SIG_SYMBOL_COL=detect_column(primary_sig,['Symbol','symbol','gene','gene_symbol','GeneSymbol'],contains=['symbol'],required=True,label='signature symbol')
primary_sig['signature_gene']=primary_sig[SIG_SYMBOL_COL].astype(str).str.strip(); primary_sig['gene_upper']=norm_symbol(primary_sig['signature_gene'])
primary_sig=primary_sig.dropna(subset=['gene_upper']).drop_duplicates('gene_upper').copy()
print('Primary signature genes:',len(primary_sig)); print(primary_sig['signature_gene'].tolist()); display(primary_sig.head(20))

## 4. Cargar candidatos priorizados de Notebook 05

In [ ]:
prioritized=pd.read_csv(PRIORITIZED_FILE)
PRIOR_SYMBOL_COL=detect_column(prioritized,['Symbol','symbol','gene','gene_symbol','GeneSymbol'],contains=['symbol'],required=True,label='prioritized symbol')
prioritized['signature_gene']=prioritized[PRIOR_SYMBOL_COL].astype(str).str.strip(); prioritized['gene_upper']=norm_symbol(prioritized['signature_gene'])
prioritized=prioritized.dropna(subset=['gene_upper']).drop_duplicates('gene_upper').copy()
print('Prioritized shape:',prioritized.shape); print('Columns:',prioritized.columns.tolist()); display(prioritized.head(20))

## 5. Construir tabla maestra de la firma

In [ ]:
signature_master=primary_sig[['signature_gene','gene_upper']].merge(prioritized,on='gene_upper',how='left',suffixes=('_primary','_prioritized'))
signature_master['signature_gene']=signature_master['signature_gene_primary']
missing=signature_master[signature_master['signature_gene_prioritized'].isna()]
print('Signature master:',signature_master.shape)
if len(missing): print('WARNING missing prioritized matches:'); display(missing[['signature_gene','gene_upper']])
else: print('✓ All primary signature genes matched prioritized candidates')
display(signature_master.head(20))

## 6. Detectar evidencia adicional de Notebook 05

In [ ]:
def optional_keywords(df, specs):
    out={}
    for label,alts in specs.items():
        hit=None
        for ks in alts:
            for c in df.columns:
                if all(k.lower() in c.lower() for k in ks): hit=c; break
            if hit: break
        out[label]=hit
    return out
nb05_cols=optional_keywords(signature_master,{
 'tier':[['tier']], 'priority_score':[['priority','score'],['candidate','score'],['score']],
 'pathway_count':[['pathway','count']], 'database_count':[['database','count'],['db','count']],
 'theme_count':[['theme','count']], 'prior_bulk_lfc':[['log2','fold'],['log2fc'],['lfc']],
 'prior_bulk_padj':[['padj'],['adjusted','p']]
})
for k,v in nb05_cols.items(): print(k,':',v)

## 7. Cargar DE formal bulk de Notebook 03

In [ ]:
bulk_de=pd.read_csv(BULK_DE_FILE)
BULK_LFC_COL=detect_column(bulk_de,['log2FoldChange','log2FC','lfc'],required=True,label='bulk log2FC')
BULK_PADJ_COL=detect_column(bulk_de,['padj','FDR','adjusted_pvalue'],required=True,label='bulk padj')
BULK_STAT_COL=detect_column(bulk_de,['stat','wald_stat','wald'])
BULK_SYMBOL_COL=detect_column(bulk_de,['Symbol','symbol','gene_symbol','GeneSymbol'],contains=['symbol'])
BULK_GENEID_COL=detect_column(bulk_de,['GeneID','gene_id','Entrez','entrez','EntrezID'],contains=['gene','id'])
MASTER_GENEID_COL=detect_column(signature_master,['GeneID','gene_id','Entrez','entrez','EntrezID'],contains=['gene','id'])
print('Bulk columns:',bulk_de.columns.tolist()); print('symbol=',BULK_SYMBOL_COL,'geneid=',BULK_GENEID_COL,'master_geneid=',MASTER_GENEID_COL)

## 8. Mapear formalmente bulk a símbolos sin asumir IDs

In [ ]:
if BULK_SYMBOL_COL is not None:
    bulk_formal=bulk_de.copy(); bulk_formal['gene_upper']=norm_symbol(bulk_formal[BULK_SYMBOL_COL]); strategy=f'direct symbol: {BULK_SYMBOL_COL}'
elif BULK_GENEID_COL is not None and MASTER_GENEID_COL is not None:
    idmap=signature_master[['gene_upper',MASTER_GENEID_COL]].dropna().copy(); idmap['merge_gene_id']=idmap[MASTER_GENEID_COL].astype(str).str.replace(r'\.0$','',regex=True).str.strip()
    bulk_formal=bulk_de.copy(); bulk_formal['merge_gene_id']=bulk_formal[BULK_GENEID_COL].astype(str).str.replace(r'\.0$','',regex=True).str.strip(); bulk_formal=bulk_formal.merge(idmap[['gene_upper','merge_gene_id']],on='merge_gene_id',how='left'); strategy='GeneID -> Notebook05 symbol'
else:
    bulk_formal=None; strategy='Notebook05 fallback only'; warnings.warn('Formal bulk DE could not be mapped safely by symbol/GeneID.')
print('Bulk mapping strategy:',strategy)

## 9. Construir evidencia bulk final y verificar cobertura

In [ ]:
bulk_evidence=signature_master[['signature_gene','gene_upper']].copy()
if bulk_formal is not None:
    cols=['gene_upper',BULK_LFC_COL,BULK_PADJ_COL]+([BULK_STAT_COL] if BULK_STAT_COL else [])
    formal=bulk_formal[cols].dropna(subset=['gene_upper']).drop_duplicates('gene_upper').copy()
    rename={BULK_LFC_COL:'bulk_log2FC',BULK_PADJ_COL:'bulk_padj'}
    if BULK_STAT_COL: rename[BULK_STAT_COL]='bulk_stat'
    formal=formal.rename(columns=rename); bulk_evidence=bulk_evidence.merge(formal,on='gene_upper',how='left')
for c in ['bulk_log2FC','bulk_padj','bulk_stat']:
    if c not in bulk_evidence.columns: bulk_evidence[c]=np.nan
pl=nb05_cols['prior_bulk_lfc']; pp=nb05_cols['prior_bulk_padj']
if pl:
    mp=signature_master[['gene_upper',pl]].drop_duplicates('gene_upper').set_index('gene_upper')[pl]; bulk_evidence['bulk_log2FC']=bulk_evidence['bulk_log2FC'].fillna(bulk_evidence['gene_upper'].map(mp))
if pp:
    mp=signature_master[['gene_upper',pp]].drop_duplicates('gene_upper').set_index('gene_upper')[pp]; bulk_evidence['bulk_padj']=bulk_evidence['bulk_padj'].fillna(bulk_evidence['gene_upper'].map(mp))
for c in ['bulk_log2FC','bulk_padj','bulk_stat']: bulk_evidence[c]=num(bulk_evidence[c])
bulk_evidence['bulk_direction']=np.where(bulk_evidence['bulk_log2FC']>0,'Up',np.where(bulk_evidence['bulk_log2FC']<0,'Down','No_change'))
bulk_evidence['bulk_FDR_lt_0.05']=bulk_evidence['bulk_padj'].notna()&(bulk_evidence['bulk_padj']<ALPHA)
n_bulk=int(bulk_evidence['bulk_log2FC'].notna().sum()); n_sig=len(bulk_evidence); mapping_fraction=n_bulk/n_sig if n_sig else 0
print(f'Mapped bulk effects: {n_bulk}/{n_sig} ({mapping_fraction:.1%})'); display(bulk_evidence)
if n_bulk < max(3,int(np.ceil(.5*n_sig))): raise RuntimeError('Too few signature genes have usable bulk effects. Stop and inspect identifiers.')

## 10. Cargar validación pseudobulk de Notebook 07

In [ ]:
sn_validation=pd.read_csv(NB07_INPUT)
required={'cell_type','signature_gene','tested','log2FoldChange','padj'}
miss=required.difference(sn_validation.columns)
if miss: raise KeyError(f'Notebook07 input missing: {sorted(miss)}')
sn_clean=sn_validation.copy(); sn_clean['gene_upper']=norm_symbol(sn_clean['signature_gene']); sn_clean['sn_log2FC']=num(sn_clean['log2FoldChange']); sn_clean['sn_padj']=num(sn_clean['padj'])
sn_clean['sn_direction']=np.where(sn_clean['sn_log2FC']>0,'Up',np.where(sn_clean['sn_log2FC']<0,'Down','No_change')); sn_clean['sn_FDR_lt_0.05']=sn_clean['sn_padj'].notna()&(sn_clean['sn_padj']<ALPHA)
sigset=set(bulk_evidence['gene_upper'].dropna()); snset=set(sn_clean['gene_upper'].dropna()); matched_sn_genes=sorted(sigset&snset)
print(f'Primary signature genes in Notebook07: {len(matched_sn_genes)}/{len(sigset)}'); print(matched_sn_genes)
if len(matched_sn_genes)<max(3,int(np.ceil(.5*len(sigset)))): raise RuntimeError('Too few signature genes represented in Notebook07.')

## 11. Integración corregida bulk × snRNA-seq

In [ ]:
cross=sn_clean.merge(bulk_evidence,on='gene_upper',how='inner',suffixes=('_sn','_bulk'))
cross['signature_gene']=cross['signature_gene_bulk']; cross['direction_concordant']=cross['sn_direction']==cross['bulk_direction']; cross['both_FDR_lt_0.05']=cross['sn_FDR_lt_0.05']&cross['bulk_FDR_lt_0.05']; cross['bulk_sig_sn_concordant']=cross['bulk_FDR_lt_0.05']&cross['direction_concordant']
cross.to_csv(NB08_RESULTS/'cross_dataset_gene_celltype_validation.csv',index=False)
n_cross_genes=int(cross['signature_gene'].nunique()); print('cross shape:',cross.shape,'genes:',n_cross_genes,'cell types:',cross['cell_type'].nunique()); print(sorted(cross['signature_gene'].dropna().unique().tolist())); display(cross.head(30))
if n_cross_genes<max(3,int(np.ceil(.5*len(primary_sig)))): raise RuntimeError('Cross-dataset merge contains too few genes; previous one-gene problem would recur.')

## 12. Tiers de validación por gen × tipo celular

In [ ]:
def tier(r):
    if r['direction_concordant'] and r['bulk_FDR_lt_0.05'] and r['sn_FDR_lt_0.05']: return 'V1_Strong_replicated'
    if r['direction_concordant'] and r['bulk_FDR_lt_0.05']: return 'V2_Directional_validation'
    if r['sn_FDR_lt_0.05'] and not r['bulk_FDR_lt_0.05']: return 'V3_Cell_type_specific'
    if r['direction_concordant']: return 'V4_Direction_only'
    return 'V5_Discordant_or_unsupported'
cross['validation_tier']=cross.apply(tier,axis=1)
cross.to_csv(NB08_RESULTS/'cross_dataset_gene_celltype_validation_tiered.csv',index=False)
display(cross.sort_values(['validation_tier','cell_type','signature_gene']))

## 13. Resumen por tipo celular

In [ ]:
celltype_summary=cross.groupby('cell_type',observed=True).agg(n_genes=('signature_gene','nunique'),n_direction_concordant=('direction_concordant','sum'),n_bulk_FDR=('bulk_FDR_lt_0.05','sum'),n_sn_FDR=('sn_FDR_lt_0.05','sum'),n_both_FDR=('both_FDR_lt_0.05','sum'),median_sn_log2FC=('sn_log2FC','median'),median_bulk_log2FC=('bulk_log2FC','median')).reset_index()
celltype_summary['fraction_direction_concordant']=celltype_summary['n_direction_concordant']/celltype_summary['n_genes']; celltype_summary['fraction_sn_FDR']=celltype_summary['n_sn_FDR']/celltype_summary['n_genes']
celltype_summary.to_csv(NB08_RESULTS/'cross_dataset_celltype_summary.csv',index=False); display(celltype_summary.sort_values(['fraction_direction_concordant','n_both_FDR'],ascending=[False,False]))

## 14. Resumen por gen

In [ ]:
gene_summary=cross.groupby('signature_gene',observed=True).agg(n_celltypes_tested=('cell_type','nunique'),n_direction_concordant=('direction_concordant','sum'),n_sn_FDR=('sn_FDR_lt_0.05','sum'),n_both_FDR=('both_FDR_lt_0.05','sum'),median_sn_log2FC=('sn_log2FC','median'),bulk_log2FC=('bulk_log2FC','first'),bulk_padj=('bulk_padj','first'),bulk_FDR=('bulk_FDR_lt_0.05','first')).reset_index()
gene_summary['fraction_direction_concordant']=gene_summary['n_direction_concordant']/gene_summary['n_celltypes_tested']; gene_summary['fraction_sn_FDR']=gene_summary['n_sn_FDR']/gene_summary['n_celltypes_tested']
gene_summary.to_csv(NB08_RESULTS/'cross_dataset_gene_summary.csv',index=False); display(gene_summary.sort_values(['fraction_direction_concordant','n_both_FDR','n_sn_FDR'],ascending=[False,False,False]))

## 15. Correlaciones y scatter plots bulk vs snRNA

In [ ]:
rows=[]
for ct,sub in cross.groupby('cell_type',observed=True):
    tmp=sub[['signature_gene','bulk_log2FC','sn_log2FC']].dropna()
    if len(tmp)<3: continue
    try: pr,pp=stats.pearsonr(tmp['bulk_log2FC'],tmp['sn_log2FC'])
    except: pr,pp=np.nan,np.nan
    try: sr,sp=stats.spearmanr(tmp['bulk_log2FC'],tmp['sn_log2FC'])
    except: sr,sp=np.nan,np.nan
    rows.append({'cell_type':ct,'n_genes':len(tmp),'pearson_r':pr,'pearson_p':pp,'spearman_rho':sr,'spearman_p':sp})
    fig,ax=plt.subplots(figsize=(5.5,5)); ax.scatter(tmp['bulk_log2FC'],tmp['sn_log2FC'],s=35,alpha=.75); ax.axhline(0,lw=.8); ax.axvline(0,lw=.8)
    for _,r in tmp.iterrows(): ax.annotate(r['signature_gene'],(r['bulk_log2FC'],r['sn_log2FC']),fontsize=7,alpha=.8)
    ax.set_xlabel('Bulk log2FC (PD / Control)'); ax.set_ylabel('snRNA pseudobulk log2FC (PD / Control)'); ax.set_title(f'Cross-dataset effects — {ct}'); fig.tight_layout(); safe=re.sub(r'[^A-Za-z0-9_.-]+','_',str(ct)); fig.savefig(NB08_FIGURES/f'bulk_vs_snRNA_{safe}.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close(fig)
correlation_table=pd.DataFrame(rows); correlation_table.to_csv(NB08_RESULTS/'cross_dataset_effect_correlations.csv',index=False); display(correlation_table)

## 16. Heatmap de concordancia de dirección

In [ ]:
direction_matrix=cross.assign(concordance_code=np.where(cross['direction_concordant'],1,-1)).pivot_table(index='signature_gene',columns='cell_type',values='concordance_code',aggfunc='first')
direction_matrix.to_csv(NB08_RESULTS/'cross_dataset_direction_concordance_matrix.csv'); display(direction_matrix)
if direction_matrix.size:
    fig,ax=plt.subplots(figsize=(max(7,direction_matrix.shape[1]*.8),max(6,direction_matrix.shape[0]*.4))); im=ax.imshow(direction_matrix.to_numpy(float),aspect='auto',vmin=-1,vmax=1); ax.set_xticks(np.arange(direction_matrix.shape[1])); ax.set_xticklabels(direction_matrix.columns,rotation=45,ha='right'); ax.set_yticks(np.arange(direction_matrix.shape[0])); ax.set_yticklabels(direction_matrix.index); ax.set_title('Direction concordance: bulk vs snRNA pseudobulk'); fig.colorbar(im,ax=ax,label='Concordance (+1 same, -1 discordant)'); fig.tight_layout(); fig.savefig(NB08_FIGURES/'direction_concordance_heatmap.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close(fig)

## 17. Heatmap de log2FC snRNA

In [ ]:
sn_lfc_matrix=cross.pivot_table(index='signature_gene',columns='cell_type',values='sn_log2FC',aggfunc='first'); sn_lfc_matrix.to_csv(NB08_RESULTS/'snRNA_signature_log2FC_matrix.csv'); display(sn_lfc_matrix)
if sn_lfc_matrix.size:
    fig,ax=plt.subplots(figsize=(max(7,sn_lfc_matrix.shape[1]*.8),max(6,sn_lfc_matrix.shape[0]*.4))); im=ax.imshow(sn_lfc_matrix.to_numpy(float),aspect='auto'); ax.set_xticks(np.arange(sn_lfc_matrix.shape[1])); ax.set_xticklabels(sn_lfc_matrix.columns,rotation=45,ha='right'); ax.set_yticks(np.arange(sn_lfc_matrix.shape[0])); ax.set_yticklabels(sn_lfc_matrix.index); ax.set_title('snRNA pseudobulk log2FC — metabolic signature'); fig.colorbar(im,ax=ax,label='log2FC PD / Control'); fig.tight_layout(); fig.savefig(NB08_FIGURES/'snRNA_signature_log2FC_heatmap.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close(fig)

## 18. Integrar evidencia adicional de Notebook 05

In [ ]:
extra=['gene_upper']+[c for c in nb05_cols.values() if c is not None and c!='gene_upper']; extra=list(dict.fromkeys(extra)); nb05_extra=signature_master[extra].drop_duplicates('gene_upper').copy(); gene_integrated=gene_summary.copy(); gene_integrated['gene_upper']=norm_symbol(gene_integrated['signature_gene']); gene_integrated=gene_integrated.merge(nb05_extra,on='gene_upper',how='left'); display(gene_integrated.head(20))

## 19. Cross-dataset validation score corregido

In [ ]:
gene_score=gene_integrated.copy()
def pct_rank_nonnegative(s):
    s=num(s)
    if s.notna().sum()==0: return pd.Series(np.zeros(len(s)),index=s.index,dtype=float)
    return s.abs().rank(pct=True,method='average').fillna(0.)
gene_score['score_direction']=num(gene_score['fraction_direction_concordant']).fillna(0).clip(0,1)
gene_score['score_sn_fdr']=num(gene_score['fraction_sn_FDR']).fillna(0).clip(0,1)
gene_score['score_bulk_fdr']=gene_score['bulk_FDR'].fillna(False).astype(float)
gene_score['score_bulk_lfc']=pct_rank_nonnegative(gene_score['bulk_log2FC']); gene_score['score_sn_lfc']=pct_rank_nonnegative(gene_score['median_sn_log2FC'])
gene_score['cross_dataset_score']=.35*gene_score['score_direction']+.25*gene_score['score_sn_fdr']+.20*gene_score['score_bulk_fdr']+.10*gene_score['score_bulk_lfc']+.10*gene_score['score_sn_lfc']
if gene_score['cross_dataset_score'].isna().any(): raise RuntimeError('NaN cross_dataset_score detected')
if (gene_score['cross_dataset_score']<0).any() or (gene_score['cross_dataset_score']>1).any(): raise RuntimeError('cross_dataset_score outside [0,1]')
gene_score=gene_score.sort_values('cross_dataset_score',ascending=False); gene_score.to_csv(NB08_RESULTS/'cross_dataset_gene_validation_score.csv',index=False)
print('gene_score shape:',gene_score.shape); print('score range:',gene_score['cross_dataset_score'].min(),gene_score['cross_dataset_score'].max()); display(gene_score[['signature_gene','cross_dataset_score','fraction_direction_concordant','fraction_sn_FDR','n_sn_FDR','n_both_FDR','bulk_log2FC','bulk_padj','median_sn_log2FC']])
if len(gene_score)<3: raise RuntimeError('Too few genes in score table')

## 20. Clasificación final por gen

In [ ]:
def classify(r):
    if r['n_both_FDR']>=1 and r['fraction_direction_concordant']>=.5: return 'Replicated'
    if bool(r['bulk_FDR']) and r['fraction_direction_concordant']>=.5: return 'Directionally_supported'
    if r['n_sn_FDR']>=1 and not bool(r['bulk_FDR']): return 'Cell_type_specific'
    if r['fraction_direction_concordant']>=.5: return 'Weak_or_mixed'
    return 'Discordant'
gene_score['final_validation_class']=gene_score.apply(classify,axis=1); gene_score.to_csv(NB08_RESULTS/'cross_dataset_gene_final_classification.csv',index=False); display(gene_score[['signature_gene','cross_dataset_score','final_validation_class','fraction_direction_concordant','n_sn_FDR','n_both_FDR','bulk_FDR']])

## 21. Gráfica corregida del score

In [ ]:
plot_df=gene_score[['signature_gene','cross_dataset_score']].dropna().sort_values('cross_dataset_score')
if len(plot_df)<3: raise RuntimeError('Too few genes to plot')
fig,ax=plt.subplots(figsize=(7,max(5,len(plot_df)*.38))); ax.barh(plot_df['signature_gene'],plot_df['cross_dataset_score']); ax.set_xlim(0,1); ax.set_xlabel('Cross-dataset validation score'); ax.set_ylabel('Gene'); ax.set_title('Cross-dataset metabolic signature validation'); fig.tight_layout(); fig.savefig(NB08_FIGURES/'cross_dataset_validation_score.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close(fig)

## 22. Exportar candidatos para Notebook 09

In [ ]:
network_cols=['signature_gene','cross_dataset_score','final_validation_class','fraction_direction_concordant','n_celltypes_tested','n_sn_FDR','n_both_FDR','bulk_log2FC','bulk_padj','median_sn_log2FC']
for c in nb05_cols.values():
    if c is not None and c in gene_score.columns and c not in network_cols: network_cols.append(c)
network_candidates=gene_score[network_cols].copy(); network_candidates['network_priority']=np.arange(1,len(network_candidates)+1); network_candidates.to_csv(NB08_RESULTS/'Notebook09_gene_network_candidates.csv',index=False); display(network_candidates)

## 23. Tabla completa de evidencia

In [ ]:
evidence_table=cross[['signature_gene','cell_type','bulk_log2FC','bulk_padj','bulk_direction','sn_log2FC','sn_padj','sn_direction','direction_concordant','bulk_FDR_lt_0.05','sn_FDR_lt_0.05','both_FDR_lt_0.05','validation_tier']].copy(); evidence_table.to_csv(NB08_RESULTS/'cross_dataset_evidence_table.csv',index=False); display(evidence_table.head(50))

## 24. Resumen automático

In [ ]:
results_summary={'n_signature_genes':int(gene_score['signature_gene'].nunique()),'n_validation_cell_types':int(cross['cell_type'].nunique())}
for cls in ['Replicated','Directionally_supported','Cell_type_specific','Weak_or_mixed','Discordant']: results_summary['n_'+cls.lower()]=int((gene_score['final_validation_class']==cls).sum())
print(json.dumps(results_summary,indent=2))

## 25. Manifest

In [ ]:
manifest08={'discovery_dataset':'GSE136666','validation_dataset':'GSE184950','discovery_modality':'bulk RNA-seq','validation_modality':'single-nucleus RNA-seq pseudobulk','tissue':'substantia nigra','primary_signature_file':str(PRIMARY_SIGNATURE_FILE),'prioritized_candidate_file':str(PRIORITIZED_FILE),'bulk_DE_file':str(BULK_DE_FILE),'snRNA_validation_file':str(NB07_INPUT),'integration_strategy':'result-level validation using Notebook05 as gene-identity master; matrices not concatenated','primary_contrast':'PD vs Control','validation_unit':'donor x cell type pseudobulk','n_signature_genes':int(gene_score['signature_gene'].nunique()),'n_validation_cell_types':int(cross['cell_type'].nunique()),'validation_classes':gene_score['final_validation_class'].value_counts().to_dict(),'top_ranked_genes':gene_score['signature_gene'].head(10).tolist(),'score_range':[float(gene_score['cross_dataset_score'].min()),float(gene_score['cross_dataset_score'].max())],'next_notebook':'09_gene_protein_enzyme_network.ipynb','caution':'Transcriptomic concordance does not establish enzyme activity or metabolic flux.'}
manifest08_path=NB08_RESULTS/'notebook08_manifest.json'; manifest08_path.write_text(json.dumps(manifest08,indent=2,ensure_ascii=False),encoding='utf-8'); print(json.dumps(manifest08,indent=2,ensure_ascii=False))

## 26. Checkpoint final corregido

In [ ]:
checks08={
'Primary signature loaded':len(primary_sig)>0,
'Notebook05 prioritized candidates loaded':len(prioritized)>0,
'Bulk DE loaded':len(bulk_de)>0,
'Bulk signature mapping sufficiently complete':mapping_fraction>=.5,
'Notebook07 validation loaded':len(sn_validation)>0,
'Multiple signature genes represented in Notebook07':len(matched_sn_genes)>=3,
'Cross-dataset integration generated':len(cross)>0,
'Multiple genes in cross-dataset integration':n_cross_genes>=3,
'At least one validation cell type':cross['cell_type'].nunique()>0,
'Gene summary generated':len(gene_summary)>0,
'Cell-type summary generated':len(celltype_summary)>0,
'No NaN cross-dataset scores':not gene_score['cross_dataset_score'].isna().any(),
'Cross-dataset scores nonnegative':(gene_score['cross_dataset_score']>=0).all(),
'Cross-dataset scores <= 1':(gene_score['cross_dataset_score']<=1).all(),
'Multiple genes in score table':len(gene_score)>=3,
'Network candidate table exported':(NB08_RESULTS/'Notebook09_gene_network_candidates.csv').exists(),
'Manifest exported':manifest08_path.exists(),
}
checkpoint08=pd.Series(checks08,name='passed').to_frame(); checkpoint08['status']=checkpoint08['passed'].map({True:'PASS',False:'FAIL'}); display(checkpoint08); checkpoint08.to_csv(NB08_RESULTS/'notebook08_final_checkpoint.csv')
failed=checkpoint08.loc[~checkpoint08['passed']]
if len(failed): print('\nSome Notebook 08 checks failed. Notebook 09 should NOT be started.'); display(failed)
else: print('\n✓ All Notebook 08 checks passed.\n\nNext: 09_gene_protein_enzyme_network.ipynb')

## 27. Interpretación y limitaciones

- El score es de priorización, no un p-value ni una probabilidad.
- La concordancia de dirección entre datasets fortalece evidencia transcriptómica.
- Bulk y snRNA-seq son estudios independientes y no deben concatenarse.
- Ausencia de significancia puede reflejar potencia limitada por donantes.
- Expresión génica no equivale a proteína, actividad enzimática ni flujo metabólico.
- Los tipos celulares deben seguir considerándose provisionales hasta validación independiente.